# Periodogram-CNN for Exoplanet Detection (Option C)

**Hypothesis (2026-07-18):** The V4 RF ceiling of 0.6066 OOF PR-AUC may be limited by
summarising each star's Lomb-Scargle periodogram into ~24 scalar features (peak power,
power concentration, periodic SNR, top-2/top-3 ratios, etc.). A CNN that operates on the
*full periodogram image* — the raw power spectrum on a fixed 500-bin frequency grid — can
learn shape features that hand-aggregation drops (harmonic stacks, secondary peak
structure, asymmetric power tails, narrow-band vs. broadband spectra).

**Design:**
- Per star: RV Lomb-Scargle on the *same 500-frequency grid* `rf_multiseed.ipynb` V4 uses.
  Comparison to V4's scalar LS features is apples-to-apples — same periodogram, different
  downstream representation (image vs. 24 hand-aggregated numbers).
- Image shape `(C, N_FREQ)` per star. **Config: CHANNELS=['rv']** for the primary run.
  Flip to `['rv','RHKp','Halpha']` in cell 1 to run the 3-channel ablation.
- Model: 4-layer Conv1d over the frequency axis, then GAP+GMP head.
- Training/eval: 10-fold StratifiedKFold × 5 reps, seeds 42-46 (identical to V4 OOF
  protocol). Threshold chosen per-rep from OOF precision-recall curve (same as V4).
- All hyperparams fixed (no per-seed tuning, no leakage from test).

**Honest comparison anchor:** V4 RF OOF PR-AUC = 0.6066 +/- 0.006. Target = 0.70.
Three outcomes:
- >= 0.70: shape features carry signal hand-aggregation lost — push further.
- > 0.6066 but < 0.70: marginal — summary ceiling is mostly real.
- <= 0.6066: information-bounded — hand-aggregation already extracts all summary features can.

## 1. Imports, configuration, data load

In [ ]:
import os
os.environ['PYTHONHASHSEED'] = '0'
import warnings
warnings.filterwarnings('ignore')

import json as _json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from astropy.timeseries import LombScargle
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_recall_curve,
    confusion_matrix, f1_score, fbeta_score, accuracy_score,
)

seed = 42
np.random.seed(seed); torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ==== CONFIG ==============================================================
CHANNELS       = ['rv']                  # ['rv','RHKp','Halpha'] for ablation
N_FREQ         = 500                      # matches rf_multiseed V4 LS grid
LS_METHOD      = 'fast'                   # astropy backend (matches V4)

# ==== OOF protocol -- IDENTICAL to rf_multiseed.ipynb cell [18] ===========
N_REPS   = 5
N_FOLDS  = 10
N_EPOCHS = 60
BATCH    = 32
LR       = 3e-4
WD       = 1e-4

# ==== Data path (Kaggle layout matches other notebooks) ===================
OBS_PKL = '/kaggle/input/datasets/maanav0114/harps-n-dataset/observations.pkl'

observations = pd.read_pickle(OBS_PKL)
print(f'Total observations: {len(observations)}')
print(f'Stars: {observations["star_name"].nunique()}')
print(f'Columns: {list(observations.columns)}')
print(f'Using device: {device}')

## 2. Star vocabulary + labels

Reproduce the alphabetical grouping from `split.py` / `baseline.ipynb` so star ordering
is identical to every other notebook (sorted `groupby('star_name')`). Labels are constant
per star by construction.

In [ ]:
grouped  = observations.groupby('star_name', sort=True)
stars    = list(grouped.groups.keys())
labels   = np.array([int(grouped.get_group(s)['has_exoplanets'].iloc[0]) for s in stars],
                    dtype=int)
n_stars  = len(stars)
print(f'n_stars = {n_stars}, positives = {int(labels.sum())}, negatives = {int((labels==0).sum())}')
print(f'Class ratio 1:{n_stars/max(labels.sum(),1):.1f}')

## 3. Per-star Lomb-Scargle periodogram on a fixed frequency grid

Each star gets `(C x N_FREQ)` — one power spectrum per channel on the SAME dense grid.
This is the 'image' the CNN sees. We compute spectra once and stack them into
`pg_array` of shape `(n_stars, C, N_FREQ)` so the OOF loop never recomputes.

The grid mirrors `rf_multiseed.ipynb`: `N_FREQ=500` frequencies spanning the dynamic
range from 1/median-baseline to 1/median-cadence. Using a SHARED grid across stars
(not star-specific) is the only way every image has the same shape for the CNN.

In [ ]:
obs_per_star = observations.groupby('star_name', sort=True).agg(
    bjd_min=('bjd','min'), bjd_max=('bjd','max'), n=('bjd','count'))
baselines = obs_per_star['bjd_max'] - obs_per_star['bjd_min']

# Low-freq cap: use the 90th pct baseline, not the median. A planet at
# period ~baseline_longest would otherwise fall off the grid and be invisible.
# 90th pct keeps all but the most extreme 10% of baselines covered.
baseline_lo = max(float(np.percentile(baselines, 90)), 1.0)
f_low  = 1.0 / baseline_lo
# High-freq cap: 1 / typical nightly cadence (median star).
typical_cadence = (baselines / obs_per_star['n'].clip(lower=2).sub(1)).median()
f_high = 1.0 / max(typical_cadence, 1.0)
if f_high <= f_low:
    f_high = f_low * 100.0  # safety net
freq_grid = np.linspace(f_low, f_high, N_FREQ)
print(f'Grid: {N_FREQ} freqs in [{f_low:.4f}, {f_high:.4f}] cyc/day')
print(f'  Periods: [{1/f_high:.2f}, {1/f_low:.2f}] days')

CHAN_COL = {'rv': 'rv_centered', 'RHKp': 'RHKp', 'Halpha': 'Halpha'}
used_cols = [CHAN_COL[c] for c in CHANNELS]
print(f'Channels ({len(CHANNELS)}): {CHANNELS} -> cols {used_cols}')

star_groups = {s: grouped.get_group(s).sort_values('bjd') for s in stars}

def compute_pgram(star):
    g = star_groups[star]
    bjd  = g['bjd'].values.astype(float)
    spectra = []
    for chan in CHANNELS:
        vals = g[CHAN_COL[chan]].values.astype(float)
        if len(bjd) < 2 or not np.all(np.isfinite(vals)) or np.std(vals) < 1e-12:
            spectra.append(np.zeros(N_FREQ, dtype=np.float32))
            continue
        ls = LombScargle(bjd, vals, fit_mean=True, center_data=True)
        power = ls.power(freq_grid, method=LS_METHOD, normalization='standard')
        spectra.append(power.astype(np.float32))
    return np.stack(spectra, axis=0)  # (C, N_FREQ)

print(f'Computing {n_stars} x {len(CHANNELS)}-channel periodograms on {N_FREQ} bins...')
star_to_pgram = {s: compute_pgram(s) for s in stars}
print('Done.')

pg_array = np.stack([star_to_pgram[s] for s in stars], axis=0).astype(np.float32)
print(f'pg_array shape: {pg_array.shape}')

# Standardize each channel grid-wise using median + MAD (robust to a few hot bins).
# LS power spectra are a feature-space transform of inputs only (no labels involved),
# so standardising using the full array is NOT label leakage.
pg_med = np.median(pg_array, axis=(0,2), keepdims=True)
pg_mad = np.median(np.abs(pg_array - pg_med), axis=(0,2), keepdims=True)
pg_std = 1.4826 * pg_mad + 1e-8
pg_array = (pg_array - pg_med) / pg_std
print(f'Standardized. mean={pg_array.mean():.4f}, std={pg_array.std():.4f}')

## 4. PyTorch Dataset + CNN model

- Dataset wraps the materialised `(n_stars, C, N_FREQ)` array with star indices. Lazy
  index lookup, NO padding — every star's image has the same fixed shape (unlike the raw
  sequence CNNs that needed variable-length padding + masking).
- Model: 4 Conv1d blocks (kernel 7 -> 5 -> 5 -> 3, channels h/2h/4h/4h), each BatchNorm
  + GELU + Dropout + 2x AvgPool. Head: GAP + GMP concat -> MLP. Wider early kernels
  catch broad-band activity; narrower later kernels catch sharp Keplerian peaks.

In [ ]:
class PgramDataset(Dataset):
    def __init__(self, idx_list, y):
        self.idx = list(idx_list)
        self.y  = y.astype(np.float32)
    def __len__(self): return len(self.idx)
    def __getitem__(self, i):
        return torch.from_numpy(pg_array[self.idx[i]]), torch.tensor(self.y[self.idx[i]]).float()


class ConvBlock(nn.Module):
    def __init__(self, ci, co, k, drop=0.3):
        super().__init__()
        self.conv = nn.Conv1d(ci, co, k, padding=k//2)
        self.bn   = nn.BatchNorm1d(co)
        self.act  = nn.GELU()
        self.drop = nn.Dropout(drop)
        self.pool = nn.AvgPool1d(2)
    def forward(self, x):
        return self.pool(self.drop(self.act(self.bn(self.conv(x)))))


class PgramCNN(nn.Module):
    def __init__(self, in_ch=1, hidden=32, drop=0.3):
        super().__init__()
        self.b1 = ConvBlock(in_ch,      hidden,    7, drop)
        self.b2 = ConvBlock(hidden,     hidden*2,  5, drop)
        self.b3 = ConvBlock(hidden*2,   hidden*4,  5, drop)
        self.b4 = ConvBlock(hidden*4,   hidden*4,  3, drop)
        self.gap = nn.AdaptiveAvgPool1d(1)
        self.gmp = nn.AdaptiveMaxPool1d(1)
        self.head = nn.Sequential(
            nn.Linear(2 * hidden*4, 32),
            nn.GELU(), nn.Dropout(drop),
            nn.Linear(32, 1),
        )
    def forward(self, x):
        h = self.b4(self.b3(self.b2(self.b1(x))))   # (B, C, L)
        a = self.gap(h).squeeze(-1)
        m = self.gmp(h).squeeze(-1)
        return self.head(torch.cat([a, m], dim=1)).squeeze(-1)  # (B,)

model = PgramCNN(in_ch=len(CHANNELS)).to(device)
n_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'PgramCNN parameters: {n_p:,}')
x0 = torch.randn(BATCH, len(CHANNELS), N_FREQ).to(device)
y0 = model(x0)
print(f'smoke forward: in={tuple(x0.shape)} out={tuple(y0.shape)}')

## 5. Training utilities

- `BCEWithLogitsLoss(pos_weight=...)` compensates the ~4:1 class imbalance.
- OneCycleLR schedule over `N_EPOCHS`.
- Threshold picked from each rep's OOF precision-recall curve (V4 protocol: F1-optimal
  threshold on held-out predictions for that rep — no leakage).

In [ ]:
def train_one_fold(train_idx, test_idx, rep_seed, verbose=False):
    """Train a fresh CNN on train_idx; return probabilities on test_idx (held-out)."""
    torch.manual_seed(rep_seed); torch.cuda.manual_seed_all(rep_seed)
    np.random.seed(rep_seed)

    model = PgramCNN(in_ch=len(CHANNELS)).to(device)
    pos = max(int((labels[train_idx]==1).sum()), 1)
    neg = max(int((labels[train_idx]==0).sum()), 1)
    pos_w = torch.tensor([neg/pos], device=device)
    crit = nn.BCEWithLogitsLoss(pos_weight=pos_w)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=LR, epochs=N_EPOCHS, steps_per_epoch=1, pct_start=0.1)

    ds_tr = PgramDataset(train_idx, labels)
    dl_tr = DataLoader(ds_tr, batch_size=BATCH, shuffle=True, drop_last=False)
    model.train()
    for ep in range(N_EPOCHS):
        for xb, yb in dl_tr:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = crit(model(xb), yb)
            loss.backward(); opt.step()
        sched.step()
        if verbose and (ep+1) % 10 == 0:
            print(f'    ep {ep+1:3d}: loss={loss.item():.4f}')

    model.eval()
    with torch.no_grad():
        x_te = torch.from_numpy(pg_array[test_idx]).to(device)
        logits = model(x_te).cpu().numpy()
    return 1.0 / (1.0 + np.exp(-logits))   # sigmoid -> probability

print('train_one_fold defined.')

## 6. OOF evaluation: 10-fold StratifiedKFold x 5 reps

**Identical protocol to `rf_multiseed.ipynb` cell [18]** so the comparison to V4 RF
0.6066 +/- 0.006 OOF PR-AUC is honest.

- `StratifiedKFold(n_splits=10, shuffle=True, random_state=42+rep)`
- Per rep: every star gets a prediction from a CNN that never saw it (out-of-fold).
- Per rep: PR-AUC, ROC-AUC, F1, F0.5, precision, recall on the OOF predictions.
- Per rep: threshold picked from the OOF PR curve (held-out, no leakage).
- Final combined OOF: average probabilities across all 5 reps.

50 total CNN fits (10 folds x 5 reps). With `N_EPOCHS=60` and ~1750 train stars per fold,
each fit takes ~30-60 s on P100/T4. Whole cell: ~25-50 min on Kaggle.

In [ ]:
all_oof_probs = np.zeros((n_stars, N_REPS))
rep_metrics   = {'rep': [], 'pr_auc': [], 'roc_auc': [],
                 'f1': [], 'f05': [], 'precision': [], 'recall': []}

print(f"{'='*72}")
print(f"OUT-OF-FOLD EVALUATION  ({len(CHANNELS)}-channel periodogram CNN)")
print(f"{'='*72}")
print(f"Protocol: {N_FOLDS}-fold StratifiedKFold x {N_REPS} reps, seeds 42-{41+N_REPS}")
print(f"CNN: 4 Conv1d blocks, N_FREQ={N_FREQ}, EPOCHS={N_EPOCHS}, BATCH={BATCH}, LR={LR}")
print(f"Per-rep fits: {N_FOLDS} | Total fits: {N_FOLDS * N_REPS}")
print()

import time
for rep in range(N_REPS):
    oof_probs = np.zeros(n_stars)
    rep_seed  = 42 + rep  # exactly mirrors V4's random_state schedule
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=rep_seed)
    fold_times = []
    for fold_idx, (train_idx, test_idx) in enumerate(skf.split(np.zeros(n_stars), labels)):
        t0 = time.time()
        probs = train_one_fold(train_idx, test_idx, rep_seed, verbose=False)
        oof_probs[test_idx] = probs
        fold_times.append(time.time() - t0)
        if (fold_idx+1) % 2 == 0:
            print(f'  rep {rep} fold {fold_idx+1:2d}/{N_FOLDS} done ({np.mean(fold_times):.0f}s/fold avg)')
        del probs
    all_oof_probs[:, rep] = oof_probs

    # per-rep metrics (same as V4 cell [18])
    roc = roc_auc_score(labels, oof_probs)
    pr  = average_precision_score(labels, oof_probs)
    vp, vr, vt = precision_recall_curve(labels, oof_probs)
    vf1 = 2 * vp * vr / (vp + vr + 1e-8)
    bi = int(np.argmax(vf1))
    thr = float(vt[bi]) if bi < len(vt) else 0.5
    preds = (oof_probs >= thr).astype(int)
    cm = confusion_matrix(labels, preds)
    tn, fp, fn, tp = cm.ravel()
    prc = tp / (tp+fp) if (tp+fp) > 0 else 0.0
    rec = tp / (tp+fn) if (tp+fn) > 0 else 0.0
    f1  = f1_score(labels, preds, zero_division=0)
    f05 = fbeta_score(labels, preds, beta=0.5, zero_division=0)
    rep_metrics['rep'].append(rep); rep_metrics['pr_auc'].append(pr)
    rep_metrics['roc_auc'].append(roc); rep_metrics['f1'].append(f1)
    rep_metrics['f05'].append(f05)
    rep_metrics['precision'].append(prc); rep_metrics['recall'].append(rec)
    print(f'  Rep {rep} seed={rep_seed}: PR-AUC={pr:.4f}  ROC={roc:.4f}  F1={f1:.4f}  F0.5={f05:.4f}  P={prc:.3f}  R={rec:.3f}  (thr={thr:.3f})')

## 7. Summary + comparison to V4 RF OOF ceiling

Anchor numbers (from `rf_multiseed.ipynb` cell [18]):
- V4 RF (OOF, 5-rep mean):    PR-AUC = 0.6066 +/- 0.006
- V4 RF (combined OOF):       PR-AUC = 0.6089
- V4 RF: ROC-AUC = 0.8459, F1 = 0.5815
- V2 test-set (split-luck inflated, NOT comparable to OOF): 0.6451
- Target: 0.70

Interpretation brackets:
- `>= 0.70`: periodogram shape carries real signal lost by scalar LS features -> push further
- `(0.6066, 0.70)`: marginal gain -> summary stats aren't the only limit
- `<= 0.6066`: information-bounded -> the RF ceiling IS the periodogram ceiling; write paper

In [ ]:
rep_df = pd.DataFrame(rep_metrics)

BASELINE = 0.6024   # original phys18 untuned, seeds 42-47
V4_PR    = 0.6066   # V4 RF OOF PR-AUC mean (rf_multiseed cell [18])
V4_PR_C  = 0.6089   # V4 combined OOF
V4_ROC   = 0.8459
V4_F1    = 0.5815
V2_TEST  = 0.6451   # V2 test-set (split-luck inflated, NOT comparable to OOF)
TARGET   = 0.70

print(f"{'='*72}")
print(f"PERIODOGRAM-CNN  ({len(CHANNELS)}-channel) -- OOF SUMMARY")
print(f"{'='*72}")
print(f"\nPer-rep PR-AUC:")
for _, row in rep_df.iterrows():
    flag = ' *** TARGET BREACHED ***' if row['pr_auc'] >= TARGET else ''
    print(f"  Rep {int(row['rep'])}: {row['pr_auc']:.4f}  ROC={row['roc_auc']:.4f}{flag}")

print(f"\nAggregate (n={N_REPS} reps):")
for m in ['pr_auc','roc_auc','f1','f05','precision','recall']:
    v = rep_df[m].values
    print(f"  {m:11s}: {v.mean():.4f} +/- {v.std(ddof=1):.4f}  (min={v.min():.4f}, max={v.max():.4f})")

# Combined OOF probabilities (averaged across reps, same as V4)
avg_oof = all_oof_probs.mean(axis=1)
combined_pr  = average_precision_score(labels, avg_oof)
combined_roc = roc_auc_score(labels, avg_oof)
vp, vr, vt = precision_recall_curve(labels, avg_oof)
vf1 = 2 * vp * vr / (vp + vr + 1e-8)
bi = int(np.argmax(vf1))
thr = float(vt[bi]) if bi < len(vt) else 0.5
combined_preds = (avg_oof >= thr).astype(int)
combined_f1   = f1_score(labels, combined_preds, zero_division=0)
combined_f05  = fbeta_score(labels, combined_preds, beta=0.5, zero_division=0)
cm = confusion_matrix(labels, combined_preds)
tn, fp, fn, tp = cm.ravel()
combined_p = tp/(tp+fp) if (tp+fp) > 0 else 0.0
combined_r = tp/(tp+fn) if (tp+fn) > 0 else 0.0

print(f"\n{'='*72}")
print(f"COMBINED OOF (probabilities averaged across {N_REPS} reps)")
print(f"{'='*72}")
print(f"  PR-AUC:  {combined_pr:.4f}")
print(f"  ROC-AUC: {combined_roc:.4f}")
print(f"  F1:      {combined_f1:.4f}  F0.5={combined_f05:.4f}  (threshold={thr:.4f})")
print(f"  P={combined_p:.3f}  R={combined_r:.3f}  (TN={tn} FP={fp} FN={fn} TP={tp})")

print(f"\n{'='*72}")
print("HEAD-TO-HEAD: periodogram-CNN vs V4 RF OOF")
print(f"{'='*72}")
print(f"                          PR-AUC    ROC-AUC    F1")
print(f"  PERIODOGRAM-CNN (rep):   {rep_df['pr_auc'].mean():.4f}    {rep_df['roc_auc'].mean():.4f}     {rep_df['f1'].mean():.4f}")
print(f"  PERIODOGRAM-CNN (comb):  {combined_pr:.4f}    {combined_roc:.4f}     {combined_f1:.4f}")
print(f"  V4 RF OOF (mean):        {V4_PR:.4f}    {V4_ROC:.4f}     {V4_F1:.4f}")
print(f"  V4 RF OOF (combined):    {V4_PR_C:.4f}")
print(f"  Baseline (phys18):       {BASELINE:.4f}")
print(f"  V2 test-set (split-luck):{V2_TEST:.4f}  (NOT OOF, NOT comparable)")
delta_pr = combined_pr - V4_PR_C
print(f"\n  Delta vs V4 combined OOF: {delta_pr:+.4f}  (positive = CNN beats RF)")
print(f"  Gap to target 0.70:      {TARGET - combined_pr:.4f}")

if combined_pr >= TARGET:
    print("\n  >>> TARGET 0.70 BREACHED -- periodogram shape carries real signal <<<<")
elif combined_pr > V4_PR_C:
    print(f"\n  >>> Periodogram CNN (+{delta_pr:.4f}) beats V4 RF. Push 3-channel ablation.")
elif combined_pr > V4_PR_C - 0.01:
    print(f"\n  >>> Tied with V4 RF (within 0.01). Periodogram shape ~= scalar LS features.")
else:
    print(f"\n  >>> CNN ({combined_pr:.4f}) trails V4 RF ({V4_PR_C:.4f}). Information-bounded -- paper.")

out_path = './cnn_periodogram_oof_results.json'
with open(out_path, 'w') as f:
    _json.dump({
        'channels': CHANNELS, 'n_freq': N_FREQ, 'n_reps': N_REPS,
        'n_folds': N_FOLDS, 'n_epochs': N_EPOCHS,
        'all_oof_probs': all_oof_probs.tolist(),
        'rep_metrics': rep_metrics, 'labels': labels.tolist(),
        'stars': stars,
        'combined_pr': combined_pr, 'combined_roc': combined_roc,
        'combined_f1': combined_f1, 'combined_f05': combined_f05,
    }, f)
print(f"\nSaved OOF results to {out_path}")
print(f"FINAL: Periodogram-CNN combined OOF PR-AUC = {combined_pr:.4f}")